# BUSI UGB-SegNet：AutoDL 手动复刻

统一使用 **conda cv / Python 3.12 / PyTorch 2.7.1 CUDA 12.8**，右上角选择 **Python (conda cv)**。
依次逐格运行：环境 → 参数 → 数据检查 → 样本 → 模型 → 训练代码 → 手动训练 → 曲线 → 完整验证 → 两套测试 → 误分图与报告。

**首次 RESUME=False；中断后 RESUME=True。没有自动启动或后台训练。** 曲线保存到磁盘，下方单元可手动查看，不加入后台刷新。

模型、损失、120 轮上限、前 50 轮部分冻结 / 第 51 轮全部解冻、AdamW 与余弦调度来自[作者代码](https://github.com/taukiralam007/UGB-SegNet/tree/20c8e4ea20691a49dd07c65cd1e3f44ef3eff2a4)。
沿用 nnU-Net 固定的 **453 / 65 / 129** 划分；不是作者 README 的 454 / 64 / 129，也不冒充作者五折结果。


## 1. 环境与路径
本地直接复用旁边 BUSI_nnUNet 的准备数据；AutoDL 完整 ZIP 自带数据和 ImageNet 骨干权重，无需重新划分或在线下载预训练权重。新机器先在终端运行 `bash Setup-AutoDL.sh`，日常无需重复安装。

In [ ]:
import os, sys, json, gc, csv, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display, Markdown, Image as NotebookImage
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
ROOT = Path.cwd().resolve()
if not (ROOT / "vendor" / "provenance.json").exists(): ROOT = ROOT / "BUSI_UGBSegNet"
assert (ROOT / "vendor" / "provenance.json").exists(), "请从 BUSI_UGBSegNet 工程目录打开 Notebook。"
assert (Path(sys.prefix) / "conda-meta").is_dir(), "请选择 conda cv 内核。"
sys.path.insert(0, str(ROOT / "scripts"))
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from common import find_data_root, manifest_rows, verify_data, write_json, MANIFEST_SHA256, UPSTREAM_COMMIT
from ugb_adapter import build_model, build_loss, BusiDataset
VERIFY_ONLY = os.environ.get("UGB_VERIFY_ONLY") == "1"
gpu = "CPU 只读验证；未验证 GPU 完整训练" if VERIFY_ONLY else ("CUDA GPU" if torch.cuda.is_available() else "当前没有 CUDA GPU")
display(pd.DataFrame({"项目": ["Python", "Conda", "PyTorch", "CUDA runtime", "GPU", "工程", "上游版本"],
                      "值": [sys.executable, sys.prefix, torch.__version__, torch.version.cuda, gpu, str(ROOT), UPSTREAM_COMMIT]}))


## 2. 参数与恢复
默认采用作者训练方案。**120 是上限；连续 25 轮没有改善时可提前停止**，best 与早停按每 5 轮的完整 224 验证 Dice 决定。每轮都额外打印验证结果供观察。

`AMP=False` 保留源码的 FP32；若以后设 True，属于显式的混合精度变体，应使用新的 RUN_NAME。当前选择规则没有 EMA。

In [ ]:
RUN_NAME = "ugb_seed42"
RESUME = False
DATA_ROOT_OVERRIDE = None  # 自定义数据目录时填写；通常无需修改
DEVICE = "cuda"
CONFIG = dict(epochs=120, batch_size=16, workers=0 if os.name == "nt" else 4,
              cpu_threads=4, phase1_end=50, lr_phase1=1e-4, lr_phase2=3e-5,
              weight_decay=1e-5, eta_min=1e-6, selection_every=5, patience=25,
              seed=42, image_size=224, threshold=.5, amp=False, save_every=1)
assert RUN_NAME.replace("_", "").replace("-", "").isalnum()
RUN_DIR = ROOT / "runs" / RUN_NAME
DATA_ROOT = find_data_root(DATA_ROOT_OVERRIDE)
manifest = pd.DataFrame(manifest_rows())
display(pd.DataFrame({"参数": list(CONFIG), "值": [str(v) for v in CONFIG.values()]}))
print("数据：", DATA_ROOT, "；输出：", RUN_DIR, "；恢复：", RESUME)


## 3. 固定数据检查
逐对检查原尺寸、0/1 标签和像素哈希。所有多掩码已合并；完全相同的图像留在同一划分，病人级独立性尚未核实。此处只做完整性检查，不进行测试集性能评价。

In [ ]:
audit = verify_data(DATA_ROOT, deep=True)
write_json(ROOT / "reports" / "data_check.json", audit)
display(manifest.groupby(["dataset", "split", "class"]).size().unstack(fill_value=0))
print("PASS：", audit["pairs"], "对；manifest SHA-256：", MANIFEST_SHA256)


### 看训练样本
固定取良性和恶性各两张；绿色是真值。网络读取时按作者方式缩到 224×224，灰度图复制为 RGB，再做 ImageNet 标准化。掩码始终使用最近邻插值，不能按图片平滑插值。

In [ ]:
import matplotlib.pyplot as plt
examples = pd.concat([manifest[(manifest["split"] == "train") & (manifest["class"] == label)].sort_values("id").head(2)
                      for label in ["benign", "malignant"]])
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for col, (_, row) in enumerate(examples.iterrows()):
    with Image.open(DATA_ROOT / row["image"]) as image: pixels = np.array(image)
    with Image.open(DATA_ROOT / row["mask"]) as image: mask = np.array(image) > 0
    axes[0, col].imshow(pixels, cmap="gray"); axes[0, col].set_title(row["id"], fontsize=9)
    axes[1, col].imshow(pixels, cmap="gray")
    axes[1, col].imshow(np.ma.masked_where(~mask, mask), cmap="Greens", alpha=.55, vmin=0, vmax=1)
    for ax in axes[:, col]: ax.axis("off")
plt.tight_layout(); plt.show()


## 4. 模型和损失
直接加载固定版本的 `UGBSegNetImproved` 和原始 `SegLoss`，不重新实现网络。EfficientNet-B0 + 三尺度 CBAM + LearnFPN + 不确定性解码器 + 三个深监督头。此单元只在 CPU 建模检查，不开始训练。

In [ ]:
preview_model = build_model(pretrained=True)
preview_loss = build_loss()
print("模型：", type(preview_model).__name__)
print("参数量：", f"{sum(p.numel() for p in preview_model.parameters()):,}")
print("损失：", preview_loss)
print("骨干权重：本工程 pretrained/，已核对 SHA-256。")
del preview_model, preview_loss
gc.collect()


## 5. 完整训练代码
下面包含训练循环、验证、两阶段优化器、原子保存和断点恢复，可直接阅读。定义函数不会启动训练。latest 每轮保存；best 在每 5 轮的选取点改善时保存；final 在正常结束或早停后保存。

In [ ]:
"""Manual single-split training using the pinned UGB-SegNet architecture and loss."""
import gc
import json
import math
import random
import time
import os
from functools import wraps
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from common import ROOT, MANIFEST_SHA256, UPSTREAM_COMMIT, manifest_rows, overlap_metrics, write_csv, write_json
from ugb_adapter import BusiDataset, build_loss, build_model, seed_worker


@dataclass
class TrainConfig:
    epochs: int = 120
    batch_size: int = 16
    workers: int = 4
    cpu_threads: int = 4
    phase1_end: int = 50
    lr_phase1: float = 1e-4
    lr_phase2: float = 3e-5
    weight_decay: float = 1e-5
    eta_min: float = 1e-6
    selection_every: int = 5
    patience: int = 25
    seed: int = 42
    image_size: int = 224
    threshold: float = .5
    amp: bool = False
    save_every: int = 1

    def contract(self):
        # Worker/thread counts are runtime choices, recorded but portable.
        return {k: v for k, v in asdict(self).items() if k not in ("workers", "cpu_threads", "save_every")}


def make_optimizer(model, config, phase):
    lr = config.lr_phase1 if phase == 1 else config.lr_phase2
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
        lr=lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,
        T_0=config.epochs if phase == 1 else config.epochs-config.phase1_end,
        T_mult=1, eta_min=config.eta_min)
    return optimizer, scheduler


def atomic_torch_save(state, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        torch.save(state, temporary)
        temporary.replace(path)
    finally:
        if temporary.exists():
            temporary.unlink()


def rng_state(device):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None}


def restore_rng(state, device):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if device.type == "cuda" and state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


@torch.inference_mode()
def validation_pass(model, criterion, loader, device, amp=False):
    model.eval()
    losses, scored = [], []
    for images, targets in loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=amp and device.type == "cuda"):
            output = model(images)
            loss = criterion(output, targets, is_ugbnet=True)
        assert torch.isfinite(loss), "Non-finite validation loss."
        losses.append(float(loss))
        probabilities = output["main"].float().sigmoid().cpu().numpy()[:, 0]
        for prob, truth in zip(probabilities, targets.cpu().numpy()[:, 0]):
            scored.append(overlap_metrics(prob > .5, truth > .5))
    assert scored
    return {"val_loss": float(np.mean(losses)), **{
        "val_"+key+"_224": float(np.mean([row[key] for row in scored]))
        for key in ("dice", "iou", "sensitivity", "specificity")}}


def save_training_curve(history, run_dir):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    epochs = [row["epoch"] for row in history]
    fig = Figure(figsize=(12, 8))
    FigureCanvasAgg(fig)
    axes = fig.subplots(2, 2)
    for key, label in (("train_loss", "train"), ("val_loss", "validation")):
        axes[0, 0].plot(epochs, [row[key] for row in history], label=label)
    axes[0, 0].set_title("Loss"); axes[0, 0].legend()
    axes[0, 1].plot(epochs, [row["val_dice_224"] for row in history], color="#0b8575")
    selected = [row for row in history if row["selection_check"]]
    axes[0, 1].scatter([row["epoch"] for row in selected], [row["val_dice_224"] for row in selected], s=18, label="selection check")
    axes[0, 1].set(title="Full validation at 224 x 224 (no EMA)", ylim=(0, 1)); axes[0, 1].legend()
    axes[1, 0].plot(epochs, [row["seconds"] for row in history]); axes[1, 0].set(title="Epoch duration", ylabel="seconds")
    axes[1, 1].plot(epochs, [row["lr_used"] for row in history]); axes[1, 1].set(title="Learning rate used", ylabel="learning rate")
    for ax in axes.flat:
        ax.set_xlabel("Completed epoch (1-based)"); ax.grid(alpha=.2)
    fig.tight_layout()
    temporary = Path(run_dir) / "progress.tmp.png"
    fig.savefig(temporary, dpi=130)
    temporary.replace(Path(run_dir) / "progress.png")
    fig.clear()


def exclusive_training(function):
    @wraps(function)
    def locked(data_root, run_dir, *args, **kwargs):
        directory = Path(run_dir).resolve()
        directory.relative_to(ROOT.resolve())
        directory.mkdir(parents=True, exist_ok=True)
        with (directory / ".training.lock").open("a+b") as lock:
            if lock.tell() == 0:
                lock.write(b"0"); lock.flush()
            lock.seek(0)
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(lock.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            return function(data_root, run_dir, *args, **kwargs)
    return locked


@exclusive_training
def train_model(data_root, run_dir, config=None, resume=False, device="cuda", *,
                smoke=False, stop_after_epoch=None, model_factory=None, rows_override=None):
    config = config or TrainConfig()
    assert config.epochs > config.phase1_end >= 1
    assert config.batch_size >= 1 and config.workers >= 0
    assert config.selection_every >= 1 and config.save_every >= 1
    assert config.image_size == 224 and config.threshold == .5
    assert config.patience >= 1
    assert not rows_override or smoke, "Partial data are allowed only in an explicitly marked smoke check."
    assert model_factory is None or smoke, "Production training must use the pinned model."
    run_dir = Path(run_dir).resolve()
    run_dir.relative_to(ROOT.resolve())
    run_dir.mkdir(parents=True, exist_ok=True)
    latest, final = run_dir / "checkpoint_latest.pth", run_dir / "checkpoint_final.pth"
    if not resume and any(run_dir.glob("checkpoint_*.pth")):
        raise FileExistsError("已有权重：设 RESUME=True，或使用新的 RUN_NAME。")
    if resume and not latest.exists():
        if final.exists() and (run_dir / "completed.json").exists():
            print("此实验已结束；可运行验证、测试或报告单元。")
            return json.loads((run_dir / "completed.json").read_text())
        raise FileNotFoundError("没有 latest 断点。首次训练请设 RESUME=False。")
    device = torch.device(device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("当前内核不能使用 CUDA；请在 AutoDL 选择 conda cv 内核。")
    torch.set_num_threads(config.cpu_threads)
    random.seed(config.seed); np.random.seed(config.seed); torch.manual_seed(config.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    checkpoint = torch.load(latest, map_location="cpu", weights_only=False) if resume else None
    if checkpoint:
        assert checkpoint["manifest_sha256"] == MANIFEST_SHA256
        assert checkpoint["upstream_commit"] == UPSTREAM_COMMIT
        assert checkpoint["smoke"] == smoke
        assert checkpoint["contract"] == config.contract(), "恢复时须保持训练计划、batch、精度和模型选择规则一致。"
        if (run_dir / "completed.json").is_file():
            completed = json.loads((run_dir / "completed.json").read_text())
            assert completed["completed_epochs"] == checkpoint["epoch"], "结束记录与 latest 不一致，请先检查该实验目录。"
            print("此实验已正常结束或早停；继续查看评价结果即可。需要新实验请换 RUN_NAME。", flush=True)
            return completed
        if checkpoint["epoch"] >= config.epochs:
            atomic_torch_save(checkpoint, final)
            result = {"status": "TRAINING_COMPLETE", "completed_epochs": checkpoint["epoch"],
                "planned_epochs": config.epochs, "best_epoch": checkpoint["best_epoch"],
                "best_val_dice_224": checkpoint["best_val_dice_224"], "smoke": smoke,
                "manifest_sha256": MANIFEST_SHA256, "upstream_commit": UPSTREAM_COMMIT}
            write_json(run_dir / "completed.json", result)
            return result
    factory = model_factory or build_model
    model = factory(pretrained=not resume).to(device)
    phase = checkpoint["phase"] if checkpoint else 1
    if phase == 1:
        model.freeze_backbone(n=4)
    else:
        model.unfreeze_all()
    optimizer, scheduler = make_optimizer(model, config, phase)
    criterion = build_loss().to(device)
    use_amp = config.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history, best, best_epoch, stale, start = [], -math.inf, None, 0, 1
    if checkpoint:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        history, best, best_epoch, stale = checkpoint["history"], checkpoint["best_val_dice_224"], checkpoint["best_epoch"], checkpoint["stale"]
        start = checkpoint["epoch"] + 1
        restore_rng(checkpoint["rng"], device)
        write_csv(run_dir / "history.csv", history)
        del checkpoint
    train_rows, val_rows = rows_override or (manifest_rows("train"), manifest_rows("val"))
    assert smoke or (len(train_rows) == 453 and len(val_rows) == 65)
    train_set = BusiDataset(train_rows, data_root, augment=True, seed=config.seed)
    val_set = BusiDataset(val_rows, data_root)
    loader_args = {"num_workers": config.workers, "pin_memory": device.type == "cuda"}
    if config.workers:
        loader_args.update(multiprocessing_context="spawn", worker_init_fn=seed_worker)
    val_loader = DataLoader(val_set, batch_size=config.batch_size, shuffle=False, **loader_args)
    write_json(run_dir / "config.json", {"config": asdict(config), "manifest_sha256": MANIFEST_SHA256,
        "upstream_commit": UPSTREAM_COMMIT, "smoke": smoke, "device": str(device),
        "selection": "macro Dice of every validation image at 224x224, every 5 epochs by default",
        "validation_logged_every_epoch": True, "data_root": str(data_root),
        "resume": "Full completed epochs only; augmentation is seeded per epoch. CUDA bitwise replay is not guaranteed."})
    early_stopped = False
    print(f"UGB-SegNet | train={len(train_rows)} val={len(val_rows)} | epochs {start}..{config.epochs} | {device} | AMP={use_amp}", flush=True)
    last_state = None
    try:
        for epoch in range(start, config.epochs + 1):
            if epoch == config.phase1_end+1 and phase == 1:
                phase = 2
                model.unfreeze_all()
                optimizer, scheduler = make_optimizer(model, config, phase)
                print(f"Phase 2: all layers unfrozen; lr={config.lr_phase2:g}", flush=True)
            tick = time.perf_counter()
            model.train()
            train_set.set_epoch(epoch)
            generator = torch.Generator().manual_seed(config.seed+epoch*1009)
            loader = DataLoader(train_set, batch_size=config.batch_size, shuffle=True,
                                generator=generator, **loader_args)
            losses = []
            lr_used = optimizer.param_groups[0]["lr"]
            for images, targets in loader:
                images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device.type, enabled=use_amp):
                    loss = criterion(model(images), targets, is_ugbnet=True)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss at epoch {epoch}.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                if not torch.isfinite(norm):
                    raise FloatingPointError(f"Non-finite gradient at epoch {epoch}.")
                scaler.step(optimizer); scaler.update()
                losses.append(float(loss.detach()))
            scheduler.step()
            validation = validation_pass(model, criterion, val_loader, device, use_amp)
            selection = epoch % config.selection_every == 0 or epoch == config.epochs
            improved = selection and validation["val_dice_224"] > best
            if selection:
                if improved:
                    best, best_epoch, stale = validation["val_dice_224"], epoch, 0
                else:
                    stale += config.selection_every
            record = {"epoch": epoch, "phase": phase, "train_loss": float(np.mean(losses)),
                **validation, "lr_used": lr_used, "lr_next": optimizer.param_groups[0]["lr"],
                "seconds": time.perf_counter()-tick, "selection_check": selection,
                "best_updated": improved, "best_val_dice_224": best if math.isfinite(best) else None}
            history.append(record)
            last_state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng": rng_state(device),
                "epoch": epoch, "phase": phase, "best_val_dice_224": best, "best_epoch": best_epoch,
                "stale": stale, "history": history, "contract": config.contract(),
                "manifest_sha256": MANIFEST_SHA256, "upstream_commit": UPSTREAM_COMMIT, "smoke": smoke}
            if epoch % config.save_every == 0 or epoch == config.epochs:
                atomic_torch_save(last_state, latest)
            if improved:
                atomic_torch_save({k: last_state[k] for k in ("model", "epoch", "phase", "best_val_dice_224", "best_epoch", "contract", "manifest_sha256", "upstream_commit", "smoke")}, run_dir / "checkpoint_best.pth")
            write_csv(run_dir / "history.csv", history)
            save_training_curve(history, run_dir)
            print(f"Epoch {epoch:03d}/{config.epochs} | phase {phase} | loss {record['train_loss']:.4f} | val_loss {record['val_loss']:.4f} | val Dice(224) {record['val_dice_224']:.4f} | lr {lr_used:.7f} | {record['seconds']:.1f}s" + (" | NEW BEST" if improved else ""), flush=True)
            if selection and stale >= config.patience:
                early_stopped = True
                break
            if stop_after_epoch is not None and epoch >= stop_after_epoch and epoch < config.epochs:
                atomic_torch_save(last_state, latest)
                print("验证用暂停：断点已保存，训练计划未改变。", flush=True)
                return {"status": "CHECKPOINT_SAVED", "epoch": epoch, "smoke": smoke}
        assert last_state is not None
        atomic_torch_save(last_state, final)
        atomic_torch_save(last_state, latest)
        result = {"status": "EARLY_STOPPED" if early_stopped else "TRAINING_COMPLETE",
            "completed_epochs": history[-1]["epoch"], "planned_epochs": config.epochs,
            "best_epoch": best_epoch, "best_val_dice_224": best, "smoke": smoke,
            "manifest_sha256": MANIFEST_SHA256, "upstream_commit": UPSTREAM_COMMIT}
        write_json(run_dir / "completed.json", result)
        print(json.dumps(result, ensure_ascii=False), flush=True)
        return result
    except KeyboardInterrupt:
        print("训练已中断；恢复使用 latest 中已保存的完整轮次。", flush=True)
        raise
    finally:
        del model, optimizer, scheduler, criterion, last_state
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


## 6. 手动开始或恢复训练
**运行这一格才会开始长时间训练。** 每轮打印训练损失、验证损失、全部 65 张的 224 Dice、学习率和耗时。这里的验证没有随机 patch 采样，也没有 EMA。

中断后将 `RESUME=True`，重新执行准备单元和本格；恢复已保存的完整轮次。改变 batch、AMP、总轮数或选取规则时另起 RUN_NAME。

In [ ]:
assert torch.cuda.is_available(), "请先在 GPU 实例上准备 CUDA 环境。"
training_result = train_model(DATA_ROOT, RUN_DIR, TrainConfig(**CONFIG), resume=RESUME, device=DEVICE)
display(training_result)


## 7. 手动查看曲线和断点
训练每轮保存 progress.png。训练暂停或结束后运行此格查看；运行中的训练占用内核，后面的单元会排队。

In [ ]:
curve_path = RUN_DIR / "progress.png"
if curve_path.exists(): display(NotebookImage(filename=str(curve_path), width=1100))
else: print("尚无已完成轮次的训练曲线。")
history_path = RUN_DIR / "history.csv"
if history_path.exists(): display(pd.read_csv(history_path).tail(10))
display(pd.DataFrame([{"权重": path.name, "MB": round(path.stat().st_size / 2**20, 1)}
                      for path in RUN_DIR.glob("checkpoint_*.pth")]))


## 8. 手动评价全部 65 张验证图
读取 best 的固定副本。同时给出原尺寸指标和 224×224 指标，逐图计算 Dice、IoU、敏感度、特异度、像素准确率、边界 HD95。固定阈值 0.5，无后处理。

原尺寸指标可与 nnU-Net 对照；输入仍是 224，概率图双线性恢复原尺寸后再阈值。此单元可在训练暂停后做中途检查。

In [ ]:
from evaluation import evaluate_checkpoint
VAL_OUTPUT = evaluate_checkpoint(DATA_ROOT, RUN_DIR, "val", device=DEVICE)
val_summary = json.loads((VAL_OUTPUT / "summary.json").read_text())
display(pd.DataFrame({grid: val_summary[grid]["lesion_macro"] for grid in ["native", "224"]}))
display(Markdown(f"[打开验证报告和误分图库]({VAL_OUTPUT.relative_to(ROOT).as_posix()}/report.html)"))


## 9. 手动内部测试：129 张
完成训练或早停后再执行。测试图不参与 best 选择。结果单独保存，不与验证结果混算。

In [ ]:
TEST_OUTPUT = evaluate_checkpoint(DATA_ROOT, RUN_DIR, "test", device=DEVICE)
test_summary = json.loads((TEST_OUTPUT / "summary.json").read_text())
display(pd.DataFrame({grid: test_summary[grid]["lesion_macro"] for grid in ["native", "224"]}))
display(Markdown(f"[打开内部测试报告]({TEST_OUTPUT.relative_to(ROOT).as_posix()}/report.html)"))


## 10. 手动外部测试：BUS-UCLM 670 张
病灶 260 张和正常图 410 张分别报告：正常图单独统计误报，不混进病灶 Dice。原始 13 张几何不匹配样本已按固定数据协议排除。外部集不参与训练或阈值调整。

In [ ]:
EXTERNAL_OUTPUT = evaluate_checkpoint(DATA_ROOT, RUN_DIR, "external", device=DEVICE)
external_summary = json.loads((EXTERNAL_OUTPUT / "summary.json").read_text())
display(pd.DataFrame({grid: external_summary[grid]["lesion_macro"] for grid in ["native", "224"]}))
display({key: external_summary["native"][key] for key in ["normal_n", "normal_false_positive_rate_any_pixel", "normal_foreground_pixels_mean"]})
display(Markdown(f"[打开外部测试报告]({EXTERNAL_OUTPUT.relative_to(ROOT).as_posix()}/report.html)"))


## 11. 选一张验证图看误分和注意力熵
设置 CASE_ID；不设置则读取验证集最低 Dice 的一张。绿色 TP、红色 FP、蓝色 FN。熵图来自源码的注意力熵，不是校准过的病灶概率。

In [ ]:
from evaluation import error_panel
CASE_ID = None
assert "VAL_OUTPUT" in globals(), "先执行第 8 节完整验证。"
scores = pd.read_csv(VAL_OUTPUT / "native_per_image.csv").sort_values("dice")
display(scores[["id", "class", "dice", "sensitivity", "fp", "fn"]].head(10))
case_id = CASE_ID or scores.iloc[0]["id"]
row = manifest[manifest["id"] == case_id].iloc[0].to_dict()
panel = VAL_OUTPUT / "error_panels" / f"{case_id}.png"
if not panel.exists(): error_panel(DATA_ROOT, row, VAL_OUTPUT / "predictions_native" / f"{case_id}.png", panel)
display(NotebookImage(filename=str(panel), width=1400))
uncertainty = VAL_OUTPUT / "uncertainty_native" / f"{case_id}.png"
if uncertainty.exists(): display(NotebookImage(filename=str(uncertainty), width=550))


## 12. 查看全部结果报告
每次评价自动写入 HTML、Markdown、JSON、逐图 CSV、原尺寸预测和误分图。下面列出已有报告，尚未运行的项目不会显示成已完成。

In [ ]:
reports = sorted((ROOT / "results" / RUN_NAME).glob("*/*/report.html"))
for report in reports:
    display(Markdown(f"[{report.parent.parent.name} · {report.parent.name}]({report.relative_to(ROOT).as_posix()})"))
if not reports: print("尚未评价；训练完成后按顺序手动运行验证、内部测试和外部测试单元。")


## 13. 使用边界与复现说明

- 这是**官方模型与损失的固定划分复刻工程**；作者发布的是五折训练驱动，本工程为了和 nnU-Net 同图比较使用单次固定划分。
- 数据读取修复：0/1 掩码用 `>0`；强制成对增强，缺少 albumentations 会报错，不启用作者未同步变换掩码的降级分支；全部多掩码已合并。
- 每轮额外验证用于观察；best 与早停默认仍每 5 轮检查。前 50 轮冻结阶段 0–3 的参数，BatchNorm 行为保留原源码。
- 224 下的 Dice / IoU 可核对作者计算网格；HD95 使用真实边界距离，单侧空掩码记未定义，区别于作者的随机前景点实现。没有毫米间距信息。
- 未运行完整 GPU 训练前，不声称复现了 README 的 81.30%。CPU 检查只证明实现可运行，不代表最终精度或远程吞吐。
- 停止按钮会中断当前单元。再次训练先确认 RESUME 和 RUN_NAME；不要同时启动两份同名实验。
